generate_data.py
* Builds the perfect Internal Ledger using Faker.
* Clones that ledger to create the Bank Statement, but purposefully "breaks" it by injecting our business anomalies (delays, fees, and missing rows).
* Exports both as CSV files into a data folder.

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import os

fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

def generate_mock_data(num_records=1000):
    print(f"Generating {num_records} internal ledger records...")
    internal_data = []
    for _ in range(num_records):
        internal_data.append({
            "Transaction_ID": fake.uuid4()[:16].upper(),
            "Date": fake.date_time_between(start_date="-30d", end_date="now").replace(microsecond=0),
            "Customer_Name": fake.name(),
            "Amount": round(random.uniform(10.0, 500.0), 2),
            "Status": "Processed"
        })
    internal_df = pd.DataFrame(internal_data)
    
    print(f"Generating {num_records} bank statement anomalies")
    bank_df = internal_df.copy()
    
    # Anomaly 1: Settlement Days Delay
    # NumPy instantly generates a list of 1,000 random numbers (e.g., [1, 3, 2, 1, 1, 3...]), the specialised np functon is much faster than looping
    # delay_days is a NumPy array, and the reason this code works so elegantly without a for loop is because of a concept called vectorization.
    # pd.to_timedelta(delay_days, unit='D') acts as a translator. It takes our NumPy array of raw numbers and converts them into specific units of time—in this cased
    delay_days = np.random.randint(1, 4, size=num_records)
    bank_df['Settlement_Date'] = bank_df['Date'] + pd.to_timedelta(delay_days, unit='D')
    bank_df = bank_df.drop(columns=['Date', 'Status'])
    bank_df.rename(columns={'Amount': 'Settled_Amount'}, inplace=True)
    
    # Anomaly 2: Apply Fees (Stripe: 2.9% + $0.30)
    fee_mask = np.random.rand(num_records) < 0.30 #around 30% of the rows will have a fee mask
    bank_df.loc[fee_mask, 'Settled_Amount'] = round(bank_df['Settled_Amount'] * (1 - 0.029) - 0.30, 2)
    
    # Anomaly 3: Missing in Bank (Drop 2% of rows to simulate failed API calls)
    drop_indices = bank_df.sample(frac=0.02, random_state=42).index
    bank_df = bank_df.drop(drop_indices)
    
    # Anomaly D: Missing Internally (Add 5 "ghost" transactions the app doesn't know about)
    ghost_data = []
    for _ in range(5):
        ghost_data.append({
            "Transaction_ID": fake.uuid4()[:16].upper(),
            "Customer_Name": fake.name().upper(), # Simulate bank capitalizing names
            "Settled_Amount": round(random.uniform(10.0, 100.0), 2),
            "Settlement_Date": fake.date_time_between(start_date="-10d", end_date="now").replace(microsecond=0)
        })
    
    # bank_df had about 1,000 rows (Index 0 to 999). We also created a tiny new DataFrame for our 5 "ghost" transactions. 
    # Because it was a brand new DataFrame, pandas automatically gave it its own index starting at 0 (Index 0 to 4).
    # When we add ignore_index=True, we are telling pandas: "Throw away the old addresses. 
    # Glue these two tables together, and then re-number the whole thing from scratch so every row has a unique, sequential number."
    bank_df = pd.concat([bank_df, pd.DataFrame(ghost_data)], ignore_index=True)
    
    # Create the data directory if it doesn't exist
    os.makedirs('../data', exist_ok=True)
    internal_df.to_csv('../data/Internal_Ledger.csv', index=False)
    bank_df.to_csv('../data/Bank_Statement.csv', index=False)
    
    print("Success! Data generated and saved to the 'data' folder.")
    
    os.makedirs('../data', exist_ok=True)
    
# if __name__ == "__main__": acts as a guard or a traffic cop for your code.
# Whenever Python runs a file, it secretly assigns a special built-in variable called __name__ to that file.
# Running the Script Directly: Python secretly set the variable __name__ = "__main__" -> It hits the if statement: "Does __name__ equal "__main__"?" Yes.
# Importing the script: Inside main.py, we will write: import generate_data -> 
    # Because it is being imported (not run directly), Python sets the __name__ variable to the file's actual name: __name__ = "generate_data".
    # It hits the if statement: "Does __name__ equal "__main__"?" No. -> Result: It skips the generate_mock_data() function entirely.
# if __name__ == "__main__":
#     generate_mock_data()
    

/var/folders/gp/9snl51jj70dd01jy_yq1bcxr0000gp/T/ipykernel_93715/2000024005.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
generate_mock_data()

Generating 1000 internal ledger records...
Generating 1000 bank statement anomalies
Success! Data generated and saved to the 'data' folder.


clean_data.py
* all about normalization—forcing both datasets to speak the exact same language before we introduce them to each other.

In [ ]:
import pandas as pd

def load_and_clean_data(internal_path, bank_path):
    print("Loading datasets...")
    # 1. Ingest the raw data
    internal_df = pd.read_csv(internal_path)
    bank_df = pd.read_csv(bank_path)

    print("Cleaning Internal Ledger...")
    # 2. Clean Internal Ledger
    # Standardize strings: Remove sneaky whitespaces and force uppercase
    internal_df['Transaction_ID'] = internal_df['Transaction_ID'].str.strip().str.upper()
    internal_df['Customer_Name'] = internal_df['Customer_Name'].str.strip().str.upper()
    
    # Standardize dates: Convert the string dates from the CSV back into actual Pandas Datetime objects
    internal_df['Date'] = pd.to_datetime(internal_df['Date'])
    
    # Ensure amounts are strict floats (decimals)
    internal_df['Amount'] = internal_df['Amount'].astype(float)

    print("Cleaning Bank Statement...")
    # 3. Clean Bank Statement
    # We apply the exact same string rules here so they match perfectly later
    bank_df['Transaction_ID'] = bank_df['Transaction_ID'].str.strip().str.upper()
    bank_df['Customer_Name'] = bank_df['Customer_Name'].str.strip().str.upper()
    
    bank_df['Settlement_Date'] = pd.to_datetime(bank_df['Settlement_Date'])
    bank_df['Settled_Amount'] = bank_df['Settled_Amount'].astype(float)

    print("Data cleaning complete!")
    return internal_df, bank_df

# The traffic cop is back! We can test this file directly.
# for the object output -> object almost always just means "String" (Text).
if __name__ == "__main__":
    internal_csv = "../data/Internal_Ledger.csv"
    bank_csv = "../data/Bank_Statement.csv"
    
    clean_internal, clean_bank = load_and_clean_data(internal_csv, bank_csv)
    
    # Let's peek at the cleaned data types to prove it worked
    print("\nInternal Ledger Data Types:")
    print(clean_internal.dtypes)
    print("\nBank Statement Data Types:")
    print(clean_bank.dtypes)

Loading datasets...
Cleaning Internal Ledger...
Cleaning Bank Statement...
Data cleaning complete!

Internal Ledger Data Types:
Transaction_ID            object
Date              datetime64[ns]
Customer_Name             object
Amount                   float64
Status                    object
dtype: object

Bank Statement Data Types:
Transaction_ID             object
Customer_Name              object
Settled_Amount            float64
Settlement_Date    datetime64[ns]
dtype: object


/var/folders/gp/9snl51jj70dd01jy_yq1bcxr0000gp/T/ipykernel_94665/2511987634.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Core Matching Logic (The Engine).
* We are going to build this engine in "Passes." In data engineering, you never try to catch everything in one massive, complex equation. 
* You filter data through a series of nets, starting with the tightest net (exact matches) and slowly using wider nets (fuzzy matches) for the leftovers.

In [ ]:

import pandas as pd
# We are importing the cleaning logic you built in Phase 2!
from clean_data import load_and_clean_data

def run_reconciliation():
    internal_csv = "../data/Internal_Ledger.csv"
    bank_csv = "../data/Bank_Statement.csv"
    internal_df, bank_df = load_and_clean_data(internal_csv, bank_csv)
    print("\n--- Starting Engine: Pass 1 (Exact Matches) ---")
    
    # Pass 1: (The Exact Match) & Isolating Leftovers
    # inner joining to find matches
    pass_1_matches = pd.merge(
        internal_df, 
        bank_df, 
        left_on=['Transaction_ID', 'Amount'], 
        right_on=['Transaction_ID', 'Settled_Amount'], 
        how='inner'
    )
    
    print(f"Pass 1: Found {len(pass_1_matches)} exact matches.")

    # The '~' symbol means "NOT". We are telling pandas: Keep rows where the ID is NOT in the matched list. We make a copy of this such that we don't mess with the original
    unmatched_internal = internal_df[~internal_df['Transaction_ID'].isin(pass_1_matches['Transaction_ID'])].copy()
    unmatched_bank = bank_df[~bank_df['Transaction_ID'].isin(pass_1_matches['Transaction_ID'])].copy()
    
    print(f"Leftovers to investigate -> Internal: {len(unmatched_internal)} | Bank: {len(unmatched_bank)}\n")
    
    # Pass 2: (The Fuzzy Match)
    print("--- Starting Engine: Pass 2 (Fuzzy Matches) ---")
    
    pass_2_matches = pd.merge(
        unmatched_internal,
        unmatched_bank,
        on='Transaction_ID',
        how='inner'
    )
    # When you subtract two pandas datetime columns (Settlement_Date - Date), pandas creates a Timedelta object (e.g., "2 days 00:00:00"). 
    # By tacking .dt.days to the end, we extract just the clean integer 2.
    pass_2_matches['Delay_Days'] = (pass_2_matches['Settlement_Date'] - pass_2_matches['Date']).dt.days

    print(f"Pass 2: Found {len(pass_2_matches)} transactions with matching IDs but different amounts.")
    
    final_missing_internal = unmatched_internal[~unmatched_internal['Transaction_ID'].isin(pass_2_matches['Transaction_ID'])].copy()
    final_missing_bank = unmatched_bank[~unmatched_bank['Transaction_ID'].isin(pass_2_matches['Transaction_ID'])].copy()
    
    print(f"Final Missing -> Internal (Failed/Dropped API): {len(final_missing_internal)}")
    print(f"Final Missing -> Bank (Ghosts/Manual Deposits): {len(final_missing_bank)}\n")
    
    print("--- Starting Engine: Phase 4 (Exception Handling from Pass 2 with matching IDs) ---")
    
    # Stripe Formula: Amount - (Amount * 2.9%) - $0.30
    expected_fee = (pass_2_matches['Amount'] * 0.029) + 0.30
    pass_2_matches['Expected_Settlement'] = pass_2_matches['Amount'] - expected_fee
    
    # Use np.isclose to compare the Expected amount vs the Actual Settled amount
    # atol=0.01 means "Absolute Tolerance of 1 cent"
    is_stripe_fee = np.isclose(
        pass_2_matches['Settled_Amount'], 
        pass_2_matches['Expected_Settlement'], 
        atol=0.01
    )
    
    pass_2_matches['Discrepancy_Reason'] = np.where(
        is_stripe_fee, 
        "Valid Stripe Fee", 
        "Unknown Discrepancy"
    )
    
    valid_fees = pass_2_matches[pass_2_matches['Discrepancy_Reason'] == "Valid Stripe Fee"]
    unknowns = pass_2_matches[pass_2_matches['Discrepancy_Reason'] == "Unknown Discrepancy"]
    
    print(f"Fee Engine: Automatically verified {len(valid_fees)} valid Stripe fees.")
    print(f"Fee Engine: Found {len(unknowns)} truly unknown discrepancies.\n")
    
    return pass_1_matches, pass_2_matches, final_missing_internal, final_missing_bank
    

# if __name__ == "__main__":
#     run_reconciliation()

export.py
* take those four separate buckets of data we isolated in Phase 4 and automatically write them into a beautiful, multi-tab Excel workbook.

In [ ]:
import pandas as pd
import os
from datetime import datetime

def generate_excel_report(pass_1, pass_2, missing_internal, missing_bank):
    print("\n--- Starting Engine: Phase 5 (Exporting to Excel) ---")
    
    # 1. Create the output directory if it doesn't exist
    os.makedirs("../output", exist_ok=True)
    
    # 2. Create a dynamic filename with today's date
    today_str = datetime.now().strftime("%Y%m%d")
    file_path = f"../output/Reconciliation_Report_{today_str}.xlsx"
    
    # 3. Open the Excel workbook in memory using openpyxl
    print(f"Writing data to {file_path}...")
    with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
        # Write each DataFrame to its own named tab (sheet_name)
        # index=False ensures we don't print those random row numbers (0, 1, 2...)
        pass_1.to_excel(writer, sheet_name='1_Reconciled_Exact', index=False)
        pass_2.to_excel(writer, sheet_name='2_Fee_Discrepancies', index=False)
        missing_internal.to_excel(writer, sheet_name='3_Missing_Internally', index=False)
        missing_bank.to_excel(writer, sheet_name='4_Missing_in_Bank', index=False)
        
    print("Success!")